# 📅 Lab W4-3 — Time Intelligence ด้วย Window Functions

**รายวิชาระบบสนับสนุนการตัดสินใจ · สัปดาห์ที่ 4 — OLAP and Multidimensional Analysis**

Lab นี้ใช้คู่กับสื่อจำลอง **Time Intelligence Builder** (`/sims/time-intelligence`)
ตัวเลขที่คุณคำนวณได้ในสมุดเล่มนี้ต้องตรงกับตัวเลขบนหน้าจอสื่อจำลองทุกหลัก

## สิ่งที่จะได้เรียนรู้
1. เขียน **YTD · MoM · YoY · Rolling 12 เดือน** ด้วย window function ได้ถูกต้อง
2. อธิบายว่าเหตุใดช่วงที่คำนวณไม่ได้ต้องเป็น **NaN ไม่ใช่ 0**
3. แสดงให้เห็นว่า **การเลือกตัววัดคือการเลือกว่าจะให้ผู้บริหารเห็นอะไร**
4. ตรวจจับ **ผลกระทบย้อนกลับ (rebound artifact)** ที่ YoY สร้างขึ้นหลังเหตุการณ์ผิดปกติ

## ข้อมูล
`sales_3years_daily.csv` — ยอดขายรายวันรายสาขา 3 ปีเต็ม (2023–2025)
มีเหตุการณ์ผิดปกติซ่อนอยู่ 1 เหตุการณ์ที่ยังไม่มีใครบอกคุณ

In [ ]:
import pandas as pd

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

URL = ("https://raw.githubusercontent.com/babankbro/ksu-dss-course/"
       "master/datasets/week04/sales_3years_daily.csv")
raw = pd.read_csv(URL)

print(f"จำนวนแถวรายวัน : {len(raw):,}")
print(f"ช่วงวันที่      : {raw.sales_date.min()} ถึง {raw.sales_date.max()}")
print(f"สาขา            : {sorted(raw.store_id.unique())}")
print(f"ภูมิภาค         : {sorted(raw.region.unique())}")
raw.head(5)

## ส่วนที่ 1 — ยุบให้เป็นระดับเดือน

ตัววัดเชิงเวลาทุกตัวคำนวณบน **grain ระดับเดือน**
ถ้ายุบผิดตั้งแต่ขั้นนี้ ตัววัดทั้ง 5 ตัวจะผิดตามทั้งหมด

### 🧑‍💻 งานที่ 1
สร้าง `DataFrame` ชื่อ `m` ที่มีหนึ่งแถวต่อหนึ่งเดือน มีคอลัมน์ `amount`
โดย **เรียงตามเวลาจากน้อยไปมาก** และมี index เป็นสตริง `YYYY-MM`

แล้วตรวจว่าได้ครบ 36 เดือนจริง และผลรวมเท่ากับผลรวมของไฟล์ดิบ

*เฉลยที่ถูกต้อง: 36 เดือน · ผลรวม 414,550,351.10 บาท*

In [ ]:
# เขียนโค้ดของคุณที่นี่


> **กับดักข้อแรก** window function ทุกตัวสมมติว่าแถวเรียงตามเวลาแล้ว
> ถ้าลืม `sort_index()` ค่า `shift()` และ `rolling()` จะเลื่อนไปผิดเดือนโดยไม่มี error ใด ๆ
> — ผลลัพธ์ผิดเงียบ ๆ ซึ่งอันตรายกว่าโปรแกรมล้ม

## ส่วนที่ 2 — ตัววัดเชิงเวลาทั้ง 5 ตัว

### 🧑‍💻 งานที่ 2
เพิ่มคอลัมน์ต่อไปนี้ลงใน `m`

| คอลัมน์ | ความหมาย | ข้อควรระวัง |
|---|---|---|
| `ytd` | ยอดสะสมตั้งแต่ต้นปี | ต้อง **รีเซ็ตทุกวันที่ 1 มกราคม** |
| `mom` | %เทียบเดือนก่อนหน้า | เดือนแรกคำนวณไม่ได้ |
| `yoy` | %เทียบเดือนเดียวกันปีก่อน | 12 เดือนแรกคำนวณไม่ได้ |
| `roll12` | ผลรวมเคลื่อนที่ 12 เดือน | 11 เดือนแรกคำนวณไม่ได้ |

**ห้ามเติม 0 ในช่องที่คำนวณไม่ได้** ให้เป็น `NaN`

*เฉลยที่ถูกต้อง: YTD ของ 2023-12 = 129,644,557.50 · YoY ของ 2024-01 = +7.49%*

In [ ]:
# เขียนโค้ดของคุณที่นี่


> **ตรวจความถูกต้องของ YTD** ค่าสุดท้ายของแต่ละปีต้องเท่ากับยอดรวมทั้งปีพอดี

In [ ]:
check = m.groupby("year").agg(ytd_last=("ytd", "last"), year_total=("amount", "sum"))
check["ตรงกัน"] = (check.ytd_last - check.year_total).abs() < 0.01
print(check.to_string())
assert check["ตรงกัน"].all(), "YTD ไม่รีเซ็ตเมื่อขึ้นปีใหม่"
print("\n✓ YTD รีเซ็ตทุกต้นปีถูกต้อง")

## ส่วนที่ 3 — เหตุการณ์ที่ซ่อนอยู่

ตอนนี้ให้หาเองว่ามีอะไรผิดปกติ โดยยังไม่ต้องอ่านคำเฉลย

### 🧑‍💻 งานที่ 3
ใช้ตัววัดที่คำนวณไว้ หา **เดือนที่ผิดปกติที่สุด** แล้วเจาะลงไปถึงระดับวัน
เพื่อระบุว่าเหตุการณ์เริ่มและจบวันที่เท่าไร

แนวทาง: เดือนที่ `yoy` ต่ำที่สุด → แล้ว plot หรือ print ยอดรายวันของเดือนนั้น

*เฉลยที่ถูกต้อง: มิถุนายน 2024 · ระบบขายขัดข้องระหว่างวันที่ 5–18*

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 4 — ตัววัดคนละตัว เล่าเรื่องคนละเรื่อง

นี่คือหัวใจของ Lab นี้ — เหตุการณ์เดียวกัน ข้อมูลชุดเดียวกัน
แต่ตัววัดที่เลือกทำให้มัน **เด่นหรือหายไป**

### 🧑‍💻 งานที่ 4
สร้างตารางเปรียบเทียบว่าเดือน 2024-06 หน้าตาเป็นอย่างไรภายใต้ตัววัดทั้ง 5 ตัว
โดยแสดงเป็น **%เบี่ยงเบนจากเดือนก่อนหน้า** ของตัววัดนั้น ๆ
เพื่อให้เทียบ "ความสะดุดตา" ข้ามตัววัดได้อย่างเป็นธรรม

แล้วตอบว่าตัววัดใดทำให้เหตุการณ์นี้ **มองไม่เห็น** และเพราะเหตุใด

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 5 — กับดักที่อันตรายกว่า: ผลย้อนกลับของ YoY

### 🧑‍💻 งานที่ 5
ดูค่า `yoy` ของเดือน **มิถุนายน 2025** แล้วอธิบายว่าเหตุใดจึงสูงผิดปกติ
และเสนอวิธีแก้ที่ทำให้รายงานปี 2025 ไม่หลอกผู้อ่าน

*เฉลยที่ถูกต้อง: YoY ของ 2025-06 = +51.79% ทั้งที่ธุรกิจไม่ได้โตขนาดนั้น*

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 6 — NaN ไม่ใช่ 0

### 🧑‍💻 งานที่ 6
สร้างสองเวอร์ชันของคอลัมน์ `yoy`

* `yoy_nan` — ปล่อยให้ 12 เดือนแรกเป็น `NaN` (ถูกต้อง)
* `yoy_zero` — เติม 0 แทน (ผิด แต่พบบ่อยมาก)

แล้วคำนวณ **ค่าเฉลี่ย YoY ตลอดช่วงข้อมูล** จากทั้งสองเวอร์ชัน
เพื่อแสดงว่าการเติม 0 บิดเบือนข้อสรุปไปกี่จุด

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 7 — เขียนเป็น SQL

### 🧑‍💻 งานที่ 7 (เขียน SQL ไม่ต้องรัน)

เขียน SQL หนึ่งคำสั่งที่คืนตารางรายเดือนพร้อมทั้ง 4 ตัววัด
โดยใช้ window function ล้วน ๆ ห้าม self-join

ข้อกำหนด
1. YTD ต้อง `PARTITION BY` ปี
2. YoY ต้องใช้ `LAG(..., 12)`
3. Rolling 12 ต้องใช้ `ROWS BETWEEN 11 PRECEDING AND CURRENT ROW`
4. ต้องคืน `NULL` (ไม่ใช่ 0) ในช่วงที่คำนวณไม่ได้

จากนั้นตอบว่า ถ้าบางเดือนไม่มียอดขายเลย (ไม่มีแถวในตาราง)
`ROWS BETWEEN` จะให้ผลผิดอย่างไร และต้องแก้ด้วยอะไร

In [ ]:
# เขียนโค้ดของคุณที่นี่


---
## ✅ เกณฑ์การส่งงาน

| องค์ประกอบ | คะแนน |
|---|:--:|
| งานที่ 1 — ยุบระดับเดือนถูกต้องและเรียงตามเวลา | 1 |
| งานที่ 2 — ตัววัดทั้ง 4 ถูกต้องและ NaN ครบตามที่ควรเป็น | 3 |
| งานที่ 3 — หาเหตุการณ์ผิดปกติและระบุช่วงวันได้ | 3 |
| งานที่ 4 — เปรียบเทียบความสะดุดตาข้ามตัววัดพร้อมคำอธิบาย | 3 |
| งานที่ 5 — อธิบายผลย้อนกลับของ YoY และเสนอวิธีแก้ | 3 |
| งานที่ 6 — แสดงผลของการเติม 0 แทน NaN ด้วยตัวเลข | 2 |
| งานที่ 7 — SQL window function ครบเงื่อนไข + ปัญหาเดือนที่ไม่มีแถว | 3 |
| **รวม** | **18** |

> 💡 ตัวเลขทุกตัวในสมุดเล่มนี้ต้องตรงกับที่แสดงบนสื่อจำลอง `/sims/time-intelligence`
> ถ้าไม่ตรง แปลว่ามีขั้นตอนใดขั้นตอนหนึ่งผิด — ให้ย้อนกลับไปตรวจก่อนส่ง